# Topic Modelling using Latent Dirichlet Allocation (LDA)

**Aim:** To apply Latent Dirichlet Allocation (LDA) to a collection of documents, discover the hidden topics and
visualise the topic–word distributions.

**Dataset:** 20 Newsgroups (11,314 training posts from 20 discussion groups).

**Steps:** preprocess → dictionary and Bag of Words → train LDA → inspect and plot topics → **test** (coherence,
choosing the number of topics, perplexity on unseen documents, topics of new documents, topics vs. true groups).

In [ ]:
!pip install -q gensim nltk pyLDAvis

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from sklearn.datasets import fetch_20newsgroups

import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

import pyLDAvis
import pyLDAvis.gensim_models

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

## 1. Load the documents

In [ ]:
# [LOAD]
newsgroups = fetch_20newsgroups(
    subset="train",
    remove=("headers", "footers", "quotes")
)

print("Number of documents :", len(newsgroups.data))
print("Number of categories:", len(newsgroups.target_names))
print("\nCategories:")
for category in newsgroups.target_names:
    print(" ", category)

print("\nFirst document:")
print(newsgroups.data[0])

In [ ]:
documents = newsgroups.data
print("Number of documents used:", len(documents))

## 2. Preprocess the text

In [ ]:
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Keep alphabetic characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords and short words
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    return tokens

processed_documents = [preprocess_text(doc) for doc in documents]

print("Original document:")
print(documents[0][:500])
print("\nProcessed document:")
print(processed_documents[0][:100])

## 3. Dictionary and Bag of Words corpus

In [ ]:
dictionary = corpora.Dictionary(processed_documents)
print("Vocabulary size:", len(dictionary))

# Remove very rare words (in fewer than 5 documents) and very common words (in more than 50% of documents)
dictionary.filter_extremes(no_below=5, no_above=0.5)
print("Vocabulary after filtering:", len(dictionary))

bow_corpus = [dictionary.doc2bow(doc) for doc in processed_documents]
print("\nFirst document BoW:")
print(bow_corpus[0][:20])

## 4. Train the LDA model

In [ ]:
num_topics = 5

lda_model = LdaModel(
    corpus=bow_corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    alpha="auto"
)

print("LDA model trained successfully!")

In [ ]:
topics = lda_model.print_topics(num_topics=num_topics, num_words=10)

for topic_id, topic_words in topics:
    print(f"\nTopic {topic_id}:")
    print(topic_words)

In [ ]:
for topic_id in range(num_topics):
    print(f"\n========== TOPIC {topic_id + 1} ==========")
    for word, probability in lda_model.show_topic(topic_id, topn=10):
        print(f"{word:<15} {probability:.4f}")

In [ ]:
topic_data = []
for topic_id in range(num_topics):
    for word, probability in lda_model.show_topic(topic_id, topn=10):
        topic_data.append({"Topic": topic_id + 1, "Word": word, "Probability": probability})

topic_df = pd.DataFrame(topic_data)
display(topic_df)

## 5. Visualise the topics

In [ ]:
for topic_id in range(num_topics):
    words = lda_model.show_topic(topic_id, topn=10)
    word_names = [word for word, probability in words]
    probabilities = [probability for word, probability in words]

    plt.figure(figsize=(10, 5))
    plt.bar(word_names, probabilities)
    plt.xlabel("Words")
    plt.ylabel("Probability")
    plt.title(f"Top Words in Topic {topic_id + 1}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
top_words = 10
heatmap_data = np.zeros((num_topics, top_words))
word_labels = []

for topic_id in range(num_topics):
    words = lda_model.show_topic(topic_id, topn=top_words)
    if topic_id == 0:
        word_labels = [word for word, probability in words]
    for i, (word, probability) in enumerate(words):
        heatmap_data[topic_id, i] = probability

plt.figure(figsize=(12, 6))
plt.imshow(heatmap_data, aspect="auto")
plt.colorbar(label="Word Probability")
plt.xlabel("Top Words")
plt.ylabel("Topics")
plt.title("Word Probabilities Across LDA Topics")
plt.show()

In [ ]:
pyLDAvis.enable_notebook()

vis = pyLDAvis.gensim_models.prepare(lda_model, bow_corpus, dictionary)
pyLDAvis.display(vis)

---
# Testing the topic model

LDA has no "right answer" column, so it is tested in four ways:

1. **Coherence** - do the top words of a topic make sense together? This is also used to choose the number of topics.
2. **Perplexity on unseen documents** - how well does the model describe documents it was not trained on?
3. **New documents** - which topics does the model assign to unseen posts and to hand-written sentences?
4. **Topics vs. true newsgroups** - the 20 Newsgroups labels are used only for checking, not for training.

## Test 1: topic coherence and the best number of topics

In [ ]:
# c_v coherence is slow on 11,000 documents, so the first 3,000 are used for the check
coherence_texts = processed_documents[:3000]

def coherence_of(model):
    return CoherenceModel(model=model, texts=coherence_texts, dictionary=dictionary,
                          coherence="c_v").get_coherence()

k_values = [3, 5, 8, 10]
rows, models = [], {}
for k in k_values:
    m = lda_model if k == num_topics else LdaModel(
        corpus=bow_corpus, id2word=dictionary, num_topics=k,
        random_state=42, passes=5, alpha="auto")
    models[k] = m
    rows.append({"Topics (K)": k, "Coherence (c_v)": round(coherence_of(m), 4)})
    print(f"K = {k:>2}  coherence = {rows[-1]['Coherence (c_v)']}")

coherence_df = pd.DataFrame(rows)
display(coherence_df)

best_k = int(coherence_df.loc[coherence_df["Coherence (c_v)"].idxmax(), "Topics (K)"])
print("Number of topics with the highest coherence:", best_k)

plt.figure(figsize=(7, 4))
plt.plot(coherence_df["Topics (K)"], coherence_df["Coherence (c_v)"], marker="o")
plt.xlabel("Number of topics (K)")
plt.ylabel("Coherence (c_v)")
plt.title("Topic coherence for different numbers of topics")
plt.xticks(k_values)
plt.show()

## Test 2: perplexity on unseen documents

In [ ]:
# [LOAD]
newsgroups_test = fetch_20newsgroups(
    subset="test",
    remove=("headers", "footers", "quotes")
)
print("Test documents available:", len(newsgroups_test.data))

In [ ]:
n_test = min(2000, len(newsgroups_test.data))
test_docs = newsgroups_test.data[:n_test]
test_targets = np.array(newsgroups_test.target[:n_test])

test_processed = [preprocess_text(d) for d in test_docs]
test_bow = [dictionary.doc2bow(d) for d in test_processed]

keep = [i for i, b in enumerate(test_bow) if len(b) > 0]            # skip documents that became empty
test_bow = [test_bow[i] for i in keep]
test_docs = [test_docs[i] for i in keep]
test_targets = test_targets[keep]
print("Unseen test documents used:", len(test_bow))

def perplexity(model, corpus):
    # gensim returns a per-word likelihood bound; perplexity = 2 ** (-bound). Lower is better.
    return 2 ** (-model.log_perplexity(corpus))

train_sample = bow_corpus[:len(test_bow)]
perp_df = pd.DataFrame({
    "Data": ["Training documents (sample)", "Unseen test documents"],
    "Perplexity": [round(perplexity(lda_model, train_sample), 1), round(perplexity(lda_model, test_bow), 1)],
})
display(perp_df)
print("Similar values on both rows mean the model describes new documents about as well as the training ones.")

## Test 3: topics of new documents

In [ ]:
def topic_distribution(model, bow):
    probs = np.zeros(model.num_topics)
    for topic, p in model.get_document_topics(bow, minimum_probability=0):
        probs[topic] = p
    return probs

rows = []
for i in range(5):
    probs = topic_distribution(lda_model, test_bow[i])
    top = int(probs.argmax())
    rows.append({
        "Newsgroup (true)": newsgroups_test.target_names[test_targets[i]],
        "Dominant topic": top + 1,
        "Probability": round(float(probs[top]), 2),
        "Text": " ".join(test_docs[i].split())[:70],
    })
display(pd.DataFrame(rows))

In [ ]:
# Hand-written sentences: does the model place them in a sensible topic?
my_texts = [
    "The team scored in the last minute and won the hockey game of the season",
    "NASA plans a new space mission to launch a satellite into orbit",
    "How do I install the driver for my graphics card on windows, the file will not open",
    "The government and the president discussed a new law about guns",
    "I believe in god and the church teaches that faith matters",
]

rows = []
for t in my_texts:
    probs = topic_distribution(lda_model, dictionary.doc2bow(preprocess_text(t)))
    top = int(probs.argmax())
    rows.append({"Sentence": t[:60], "Dominant topic": top + 1, "Probability": round(float(probs[top]), 2)})
display(pd.DataFrame(rows))

# Show the distribution of the first sentence over all topics
probs = topic_distribution(lda_model, dictionary.doc2bow(preprocess_text(my_texts[0])))
plt.figure(figsize=(6, 3.5))
plt.bar([f"Topic {i + 1}" for i in range(num_topics)], probs)
plt.ylabel("Probability")
plt.title("Topic mixture of: " + my_texts[0][:45] + "...")
plt.show()

## Test 4: do the topics match the real newsgroups?

In [ ]:
dominant = np.array([int(topic_distribution(lda_model, b).argmax()) for b in test_bow])

crosstab = pd.crosstab(
    pd.Series([newsgroups_test.target_names[t] for t in test_targets], name="True newsgroup"),
    pd.Series(dominant + 1, name="LDA topic"),
)

plt.figure(figsize=(8, 8))
sns.heatmap(crosstab.div(crosstab.sum(axis=1), axis=0) * 100, annot=True, fmt=".0f", cmap="Blues",
            cbar_kws={"label": "% of the newsgroup's documents"})
plt.title("Which LDA topic does each newsgroup fall into?")
plt.tight_layout()
plt.show()

# Purity: for every topic take the most common newsgroup; share of documents that agree with it
purity = crosstab.max(axis=0).sum() / crosstab.values.sum()
print(f"Topic purity: {purity * 100:.1f}%   (random guessing with 20 groups would be about 5%)")

summary = []
for topic in crosstab.columns:
    col = crosstab[topic].sort_values(ascending=False)
    summary.append({"LDA topic": topic, "Documents": int(col.sum()),
                    "Most common newsgroup": col.index[0],
                    "Share": f"{col.iloc[0] / col.sum() * 100:.0f}%"})
display(pd.DataFrame(summary))

## Conclusion

Documents from the 20 Newsgroups collection were cleaned, converted to a Bag of Words corpus and modelled with
LDA. Each topic is a probability distribution over words and was visualised with bar charts, a heat map and
pyLDAvis. The testing section measured topic coherence for different numbers of topics, perplexity on unseen
posts, the topics of new documents and sentences, and how well the discovered topics agree with the real
newsgroups.